# IPL Match Prediction — Jupyter Notebook

This notebook is the Jupyter-friendly version of the uploaded IPL Prediction Website project. It uses the **original Random Forest model** and the original `players.csv` and `matches.csv` datasets extracted from the project ZIP.

**Files required in the same folder as this notebook**
- `ipl_prediction_model.pkl` — original trained model
- `players.csv` — player/team performance data
- `matches.csv` — model training/evaluation data

> The original Streamlit website is not required to run these notebook cells. This notebook provides data loading, analysis, match prediction, player lookup, and team comparison directly in Jupyter.


## 1. Install required packages
Run this cell if the libraries are not already installed.

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn ipywidgets


## 2. Import libraries and load project files

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

DATA_DIR = Path(".")  # Change this if your CSV/model files are stored elsewhere
MODEL_PATH = DATA_DIR / "ipl_prediction_model.pkl"
PLAYERS_PATH = DATA_DIR / "players.csv"
MATCHES_PATH = DATA_DIR / "matches.csv"

for path in [MODEL_PATH, PLAYERS_PATH, MATCHES_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path.resolve()}\n"
            "Place the model and both CSV files beside this notebook."
        )

players = pd.read_csv(PLAYERS_PATH)
matches = pd.read_csv(MATCHES_PATH)

with MODEL_PATH.open("rb") as f:
    model = pickle.load(f)

print("Files loaded successfully.")
print(f"Players: {len(players):,}")
print(f"Match records: {len(matches):,}")
print(f"Model: {type(model).__name__}")
print(f"Model input features: {getattr(model, 'n_features_in_', 'unknown')}")


## 3. Inspect and validate the datasets

In [ ]:
print("PLAYER DATA")
display(players.head())

print("\nMATCH DATA")
display(matches.head())

required_player_columns = {"Player_Name", "Team", "Performance_Percentage"}
required_match_columns = {
    "Team1_Strength", "Team2_Strength",
    "Toss_Advantage", "Venue_Factor", "Team1_Win"
}

missing_player = required_player_columns - set(players.columns)
missing_match = required_match_columns - set(matches.columns)

if missing_player:
    raise ValueError(f"players.csv is missing columns: {sorted(missing_player)}")
if missing_match:
    raise ValueError(f"matches.csv is missing columns: {sorted(missing_match)}")

print("\nDataset validation passed.")
print("Teams:", sorted(players["Team"].dropna().unique()))


## 4. Calculate team strengths
The original Streamlit application calculates each team's strength as the mean `Performance_Percentage` of its players.

In [ ]:
team_strengths = (
    players.groupby("Team")["Performance_Percentage"]
    .mean()
    .sort_values(ascending=False)
)

team_summary = team_strengths.rename("Team_Strength").reset_index()
team_summary["Team_Strength"] = team_summary["Team_Strength"].round(2)
display(team_summary)

plt.figure(figsize=(11, 5))
team_strengths.sort_values().plot(kind="barh")
plt.title("Average Player Performance by IPL Team")
plt.xlabel("Average Performance Percentage")
plt.ylabel("Team")
plt.tight_layout()
plt.show()

teams = sorted(players["Team"].dropna().unique().tolist())


## 5. Match prediction function
This function follows the exact input feature order used by the original application: `Team1_Strength`, `Team2_Strength`, `Toss_Advantage`, `Venue_Factor`.

In [ ]:
FEATURE_COLUMNS = [
    "Team1_Strength",
    "Team2_Strength",
    "Toss_Advantage",
    "Venue_Factor",
]

def predict_match(team1, team2, venue="Neutral", toss_winner=None):
    """Predict the likely winner using the project's original trained model.

    venue: 'Neutral', 'Team 1 Home', or 'Team 2 Home'
    toss_winner: exact team name; defaults to team1
    """
    if team1 not in team_strengths.index:
        raise ValueError(f"Unknown Team 1: {team1}")
    if team2 not in team_strengths.index:
        raise ValueError(f"Unknown Team 2: {team2}")
    if team1 == team2:
        raise ValueError("Please select two different teams.")

    if venue not in {"Neutral", "Team 1 Home", "Team 2 Home"}:
        raise ValueError("venue must be Neutral, Team 1 Home, or Team 2 Home")

    if toss_winner is None:
        toss_winner = team1
    if toss_winner not in {team1, team2}:
        raise ValueError("toss_winner must be either team1 or team2")

    strength1 = float(team_strengths[team1])
    strength2 = float(team_strengths[team2])

    # These values intentionally match the original app.py logic.
    venue_factor = {
        "Neutral": 0.0,
        "Team 1 Home": 3.0,
        "Team 2 Home": -3.0,
    }[venue]

    toss_advantage = 2.0 if toss_winner == team1 else -2.0

    input_data = pd.DataFrame(
        [[strength1, strength2, toss_advantage, venue_factor]],
        columns=FEATURE_COLUMNS
    )

    probabilities = model.predict_proba(input_data)[0]
    classes = list(model.classes_)

    # The original app treats class 1 as Team 1 winning and class 0 as Team 2 winning.
    probability_team1 = float(probabilities[classes.index(1)])
    probability_team2 = float(probabilities[classes.index(0)])

    winner = team1 if probability_team1 >= probability_team2 else team2
    confidence = max(probability_team1, probability_team2)

    return {
        "Team 1": team1,
        "Team 2": team2,
        "Team 1 Strength": round(strength1, 2),
        "Team 2 Strength": round(strength2, 2),
        "Venue": venue,
        "Toss Winner": toss_winner,
        "Team 1 Win Probability": round(probability_team1 * 100, 2),
        "Team 2 Win Probability": round(probability_team2 * 100, 2),
        "Predicted Winner": winner,
        "Model Confidence": round(confidence * 100, 2),
    }

print("Prediction function is ready.")


## 6. Make a match prediction
Edit the team names and match conditions below. Use exact team names printed in the team list.

In [ ]:
# Example inputs — change these values to predict another match.
team1 = teams[0]
team2 = teams[1]
venue = "Neutral"          # Neutral | Team 1 Home | Team 2 Home
toss_winner = team1        # Must be team1 or team2

prediction = predict_match(team1, team2, venue, toss_winner)
display(pd.DataFrame([prediction]))

print(
    f"Predicted winner: {prediction['Predicted Winner']} "
    f"({prediction['Model Confidence']:.1f}% model confidence)"
)


## 7. Interactive prediction controls (optional)
If `ipywidgets` is available in your Jupyter environment, use the dropdowns to predict without editing Python variables.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    team1_widget = widgets.Dropdown(options=teams, description="Team 1:")
    team2_widget = widgets.Dropdown(
        options=[t for t in teams if t != team1_widget.value],
        description="Team 2:"
    )
    venue_widget = widgets.Dropdown(
        options=["Neutral", "Team 1 Home", "Team 2 Home"],
        description="Venue:"
    )
    toss_widget = widgets.Dropdown(
        options=[team1_widget.value, team2_widget.value],
        description="Toss won:"
    )
    predict_button = widgets.Button(description="Predict Winner", button_style="success")
    output = widgets.Output()

    def update_options(change=None):
        available = [t for t in teams if t != team1_widget.value]
        if team2_widget.value == team1_widget.value:
            team2_widget.value = available[0]
        team2_widget.options = available
        toss_widget.options = [team1_widget.value, team2_widget.value]
        if toss_widget.value not in toss_widget.options:
            toss_widget.value = team1_widget.value

    team1_widget.observe(update_options, names="value")
    team2_widget.observe(lambda change: setattr(toss_widget, "options",
                                                [team1_widget.value, team2_widget.value]), names="value")

    def on_predict_clicked(_):
        with output:
            clear_output(wait=True)
            result = predict_match(
                team1_widget.value,
                team2_widget.value,
                venue_widget.value,
                toss_widget.value
            )
            display(pd.DataFrame([result]))
            print(f"🏆 Predicted winner: {result['Predicted Winner']}")
            print(f"Model confidence: {result['Model Confidence']:.1f}%")

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([
        widgets.HBox([team1_widget, team2_widget]),
        widgets.HBox([venue_widget, toss_widget]),
        predict_button,
        output
    ]))
except ImportError:
    print("ipywidgets is not installed. Run: %pip install ipywidgets")


## 8. Player performance lookup

In [ ]:
def show_player_performance(player_name):
    result = players[players["Player_Name"].str.contains(
        player_name, case=False, na=False
    )]
    if result.empty:
        print(f"No player found matching: {player_name}")
    else:
        display(result)

# Example: replace with a player name or part of a name.
show_player_performance(players.iloc[0]["Player_Name"])


## 9. Compare two teams

In [ ]:
def compare_teams(team_a, team_b):
    if team_a == team_b:
        raise ValueError("Choose two different teams.")
    comparison = players[players["Team"].isin([team_a, team_b])].groupby("Team").agg(
        Players=("Player_Name", "count"),
        Average_Performance=("Performance_Percentage", "mean"),
        Average_Batting_Rating=("Batting_Rating", "mean"),
        Average_Bowling_Rating=("Bowling_Rating", "mean"),
        Total_Runs=("Runs", "sum"),
        Total_Wickets=("Wickets", "sum"),
    ).round(2)
    display(comparison)

    comparison[["Average_Performance", "Average_Batting_Rating",
                "Average_Bowling_Rating"]].plot(kind="bar", figsize=(10, 5))
    plt.title(f"Team Comparison: {team_a} vs {team_b}")
    plt.ylabel("Average rating / percentage")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

compare_teams(teams[0], teams[1])


## 10. Explore the match dataset and model performance
This evaluates the saved model on the supplied `matches.csv` records. The dataset appears to be a project dataset; treat the metrics as dataset-specific rather than proof of real-world IPL accuracy.

In [ ]:
X = matches[FEATURE_COLUMNS]
y = matches["Team1_Win"].astype(int)

if hasattr(model, "score"):
    accuracy = model.score(X, y)
    print(f"Accuracy on supplied matches.csv: {accuracy:.3f} ({accuracy * 100:.1f}%)")

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

predicted = model.predict(X)
print(classification_report(y, predicted, zero_division=0))

cm = confusion_matrix(y, predicted, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Team 2 win (0)", "Team 1 win (1)"]).plot()
plt.title("Confusion Matrix — Supplied Match Dataset")
plt.tight_layout()
plt.show()


## 11. Save prediction results (optional)


In [ ]:
# Save the most recent example prediction to CSV.
pd.DataFrame([prediction]).to_csv("ipl_prediction_result.csv", index=False)
print("Saved: ipl_prediction_result.csv")


## Notes and limitations

- The notebook uses the **original model and data files from the uploaded ZIP**.
- Predictions use the same team-strength, toss, and venue-factor rules as the original Streamlit app.
- Model confidence is the classifier's predicted class probability, not a guarantee of the match outcome.
- The supplied dataset is a project dataset. Do not interpret its scores as independently validated real-world IPL forecasting accuracy.
- The original Streamlit interface can still be launched separately with `streamlit run app.py` from the original project folder.
